In [1]:
import random
size = 100
matrix = [[0 for _ in range(size)] for _ in range(size)]

for i in range(size):
    for j in range(size):
        matrix[i][j] = round(random.uniform(10, 90), 2)

In [2]:
def create_population(population_size: int):
    population = []
    for _ in range(population_size):
        genome_to_shuffle = [i for i in range(0, size)]
        random.shuffle(genome_to_shuffle)
        population.append(genome_to_shuffle)
    return population


def fitness(genes: list):
    total_distance = 0
    for i in range(len(genes) - 1):
        total_distance += matrix[genes[i]][genes[i + 1]]
    total_distance += matrix[genes[-1]][genes[0]]
    return total_distance


def swap_mutation(genes: list):
    new_genes = list(genes)
    idx1, idx2 = random.sample(range(len(genes)), 2)
    new_genes[idx1], new_genes[idx2] = new_genes[idx2], new_genes[idx1]
    return new_genes


def inversion_mutation(genome: list):
    size = len(genome)
    start, end = sorted(random.sample(range(size), 2))
    reversed_fragment = genome[start:end + 1][::-1]
    genome[start:end + 1] = reversed_fragment
    return genome


def crossover_ox(parent1: list, parent2: list, inversion_chance, swap_chance):
    size = len(parent1)
    child = [-1] * size

    start, end = sorted(random.sample(range(size), 2))
    child[start:end + 1] = parent1[start:end + 1]

    fragment_set = set(child[start:end + 1])
    genes_from_parent2 = [gene for gene in parent2 if gene not in fragment_set]

    pointer = 0
    for i in range(size):
        if child[i] == -1:
            child[i] = genes_from_parent2[pointer]
            pointer += 1

    if random.random() < inversion_chance:
        child = inversion_mutation(child)

    if random.random() < swap_chance:
        child = swap_mutation(child)

    return child


def next_generation(population: list, inversion_chance, swap_chance):
    target_size = len(population)

    costs = [(fitness(genes), genes) for genes in population]
    costs.sort(key=lambda x: x[0])
    best_result = costs[0][0]
    best_genome = costs[0][1]

    sorted_population = [genes for _, genes in costs]

    elite = sorted_population[:int(0.05 * target_size)]
    breeding_pool = sorted_population[:int(0.50 * target_size)]

    new_population = list(elite)

    while len(new_population) < target_size:
        lucky_parent1, lucky_parent2 = random.sample(breeding_pool, 2)
        new_child = crossover_ox(lucky_parent1, lucky_parent2, inversion_chance, swap_chance)
        new_population.append(new_child)

    return new_population[:target_size], best_result, best_genome

In [3]:
global_best_cost = float('inf')
global_best_genome = []

num_runs = 5
num_generations = 2000

for run in range(num_runs):
    print(f"\n" + "=" * 40)
    print(f"START {run + 1} / {num_runs}")
    print("=" * 40)

    population = create_population(1000)
    round_best_cost = float('inf')
    round_best_genome = []

    generations_without_improvement = 0
    patience_limit = 70

    for i in range(num_generations):
        inversion_chance = 0.10
        swap_chance = 0.02

        if generations_without_improvement >= patience_limit:
            inversion_chance = 0.40
            swap_chance = 0.30
            generations_without_improvement = 0

        population, cost, best_route = next_generation(population, inversion_chance, swap_chance)

        if cost < round_best_cost:
            round_best_cost = cost
            round_best_genome = best_route
            generations_without_improvement = 0
        else:
            generations_without_improvement += 1

        if i % 500 == 0:
            print(f"Run {run + 1} | Generation {i}: Cost = {round(cost, 2)}")

    print(f"\n End of run {run + 1} Best result: {round(round_best_cost, 2)}")

    if round_best_cost < global_best_cost:
        global_best_cost = round_best_cost
        global_best_genome = round_best_genome
        print("NEW GLOBAL RECORD")

print("\n" + "*" * 40)
print("=== FINAL RESULTS ===")
print("*" * 40)
print(f"Best route found: {global_best_genome}")
print(f"Final minimum distance: {round(global_best_cost, 2)}")


START 1 / 5
Run 1 | Generation 0: Cost = 4210.17
Run 1 | Generation 500: Cost = 1713.43
Run 1 | Generation 1000: Cost = 1694.51
Run 1 | Generation 1500: Cost = 1694.51

 End of run 1 Best result: 1606.63
NEW GLOBAL RECORD

START 2 / 5
Run 2 | Generation 0: Cost = 4281.37
Run 2 | Generation 500: Cost = 1722.77
Run 2 | Generation 1000: Cost = 1654.4
Run 2 | Generation 1500: Cost = 1635.4

 End of run 2 Best result: 1635.4

START 3 / 5
Run 3 | Generation 0: Cost = 4200.1
Run 3 | Generation 500: Cost = 1792.38
Run 3 | Generation 1000: Cost = 1775.23
Run 3 | Generation 1500: Cost = 1768.86

 End of run 3 Best result: 1768.86

START 4 / 5
Run 4 | Generation 0: Cost = 4332.04
Run 4 | Generation 500: Cost = 1706.84
Run 4 | Generation 1000: Cost = 1624.31
Run 4 | Generation 1500: Cost = 1607.36

 End of run 4 Best result: 1605.45
NEW GLOBAL RECORD

START 5 / 5
Run 5 | Generation 0: Cost = 4256.86
Run 5 | Generation 500: Cost = 1735.97
Run 5 | Generation 1000: Cost = 1675.26
Run 5 | Generation 